In [1]:
"""
GMM + SMOTE Augmentation
─────────────────────────
Step 1: GMM generates ~300 synthetic rows (preserves real patterns)
Step 2: SMOTE balances Group distribution (1 / 2 / 3)
Final output: ~500 rows, balanced across groups, same 29 columns

Command: pip install imbalanced-learn
         python gmm_smote_augment.py
"""

import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

GMM_TARGET   = 250   # rows after GMM step
FINAL_TARGET = 500   # rows after SMOTE step

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv("../Dataset/Cleaned_Data.csv")
print(f"✅ Original: {len(df)} rows")
print(f"   Group dist: {df['Group'].value_counts().sort_index().to_dict()}\n")

# ── Column definitions ────────────────────────────────────────────────────────
ORDINAL = {
    "Place_Grew_Up":      [0.0, 1.0, 2.0, 3.0],
    "Price_Importance":   [0.0, 1.0, 2.0],
    "Brand_Importance":   [0.0, 1.0, 2.0],
    "Peer_Importance":    [0.0, 1.0, 2.0],
    "Utility_Importance": [0.0, 1.0, 2.0],
}
OHE_TRACK = ["Track_Bank_Balance", "Track_None", "Track_Payment_Apps", "Track_Apps_Spreadsheet"]
OHE_GRAPH = ["Graph_Irregular_Random", "Graph_Spike_Then_Low", "Graph_High_Weekends",
             "Graph_Uniform_Daily", "Expenditure_Graph_nan"]
BINARY    = ["Emergency(JBS)", "Discounts(JBS)", "Party(JBS)", "Workshop(JBS)", "Trip(JBS)",
             "Budget_FoodDining", "Budget_Travel", "Budget_Fashion",
             "Budget_Subscriptions", "Budget_Entertainment"]
SCALED      = ["Unplanned_Purchases", "Peer_Influence", "Finance_Confidence"]
SCALED_VALS = np.array([0.0, 0.25, 0.5, 0.75, 1.0])

def snap(val):
    return SCALED_VALS[np.argmin(np.abs(SCALED_VALS - np.clip(val, 0.0, 1.0)))]

def get_group(spend):
    if spend <= 3:   return 1
    elif spend <= 6: return 2
    else:            return 3

def fix_synthetic(df_s):
    """Snap all columns to valid discrete values."""
    for col, valid in ORDINAL.items():
        arr = np.array(valid)
        df_s[col] = df_s[col].apply(lambda v: arr[np.argmin(np.abs(arr - np.clip(v, arr[0], arr[-1])))])
    for group in [OHE_TRACK, OHE_GRAPH]:
        vals = df_s[group].values.astype(float)
        ohe = np.zeros_like(vals)
        ohe[np.arange(len(vals)), np.argmax(vals, axis=1)] = 1.0
        df_s[group] = ohe
    for col in BINARY:
        df_s[col] = np.round(df_s[col]).clip(0, 1).astype(int)
    for col in SCALED:
        df_s[col] = df_s[col].apply(snap)
    df_s["Monthly_Spend"] = np.round(df_s["Monthly_Spend"]).clip(1, 10).astype(int)
    return df_s

# ════════════════════════════════════════════════════════
# STEP 1 — GMM: 123 → 300 rows
# ════════════════════════════════════════════════════════
df_num = df.drop(columns=["Group"])
SYNTHETIC_NEEDED = GMM_TARGET - len(df)

gmm = GaussianMixture(
    n_components=min(8, len(df_num) // 12),
    covariance_type="full",
    random_state=42,
    max_iter=300,
    n_init=5,
)
gmm.fit(df_num.values)
print(f"✅ GMM fitted | components = {gmm.n_components}")

raw, _ = gmm.sample(SYNTHETIC_NEEDED * 4)
df_syn = pd.DataFrame(raw, columns=df_num.columns)
df_syn = fix_synthetic(df_syn)

# Keep highest-likelihood
scores   = gmm.score_samples(df_syn.values)
df_syn   = (df_syn.assign(__score__=scores)
                  .sort_values("__score__", ascending=False)
                  .head(SYNTHETIC_NEEDED)
                  .drop(columns=["__score__"])
                  .reset_index(drop=True))
df_syn["Group"] = df_syn["Monthly_Spend"].apply(get_group)

df_gmm = pd.concat([df, df_syn], ignore_index=True).reset_index(drop=True)
print(f"✅ After GMM: {len(df_gmm)} rows")
print(f"   Group dist: {df_gmm['Group'].value_counts().sort_index().to_dict()}\n")

# ════════════════════════════════════════════════════════
# STEP 2 — SMOTE: balance Group 1 / 2 / 3 → ~500 rows
# ════════════════════════════════════════════════════════

# SMOTE needs purely numeric input — Group col is the label
X_smote = df_gmm.drop(columns=["Group"]).values.astype(float)
y_smote = df_gmm["Group"].values

# Target: equal counts per group summing to ~FINAL_TARGET
per_class = FINAL_TARGET // 3
sampling_strategy = {
    1: max(per_class, df_gmm[df_gmm["Group"]==1].shape[0]),
    2: per_class,
    3: per_class,
}

smote = SMOTE(
    sampling_strategy=sampling_strategy,
    random_state=42,
    k_neighbors=min(5, df_gmm[df_gmm["Group"]==3].shape[0] - 1),  # safe k
)
X_bal, y_bal = smote.fit_resample(X_smote, y_smote)

df_final = pd.DataFrame(X_bal, columns=df_gmm.drop(columns=["Group"]).columns)
df_final["Group"] = y_bal

# SMOTE may produce non-discrete values → snap again
df_final = fix_synthetic(df_final)
df_final["Group"] = df_final["Monthly_Spend"].apply(get_group)  # re-derive for consistency

# Shuffle
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

# Restore dtypes
float_cols = list(ORDINAL.keys()) + OHE_TRACK + OHE_GRAPH + SCALED
int_cols   = BINARY + ["Monthly_Spend", "Group"]
for col in float_cols: df_final[col] = df_final[col].astype(float)
for col in int_cols:   df_final[col] = df_final[col].astype(int)

# Match original column order
df_final = df_final[df.columns]

# ── Results ───────────────────────────────────────────────────────────────────
print(f"✅ After SMOTE: {len(df_final)} rows")
print(f"   Group dist:  {df_final['Group'].value_counts().sort_index().to_dict()}")
print(f"   Monthly_Spend dist: {df_final['Monthly_Spend'].value_counts().sort_index().to_dict()}")
print(f"   Column names match: {list(df_final.columns) == list(df.columns)}")

# ── Save ──────────────────────────────────────────────────────────────────────
df_final.to_csv("../Dataset/Augmented_Cleaned_Data.csv", index=False)
print(f"\n🎉 Saved → ../Dataset/Augmented_Cleaned_Data.csv")


✅ Original: 123 rows
   Group dist: {1: 58, 2: 42, 3: 23}

✅ GMM fitted | components = 8
✅ After GMM: 250 rows
   Group dist: {1: 175, 2: 52, 3: 23}

✅ After SMOTE: 507 rows
   Group dist:  {1: 175, 2: 166, 3: 166}
   Monthly_Spend dist: {1: 38, 2: 66, 3: 71, 4: 85, 5: 53, 6: 28, 7: 12, 8: 19, 9: 50, 10: 85}
   Column names match: True

🎉 Saved → ../Dataset/Augmented_Cleaned_Data.csv


In [2]:
"""
FILE 2: Reverse Decode Augmented_Cleaned_Data.csv → Categorical strings
─────────────────────────────────────────────────────────────────────────
Converts the integer/encoded Augmented_Cleaned_Data.csv back to original
categorical string format needed by K-Prototypes.

Also inverse-transforms MinMaxScaler cols back to original 1–5 scale.

Command: python gmm_reverse_decode.py
Output:  ../data/Augmented_KProto_Data.csv
"""

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

# ── Load augmented cleaned data ───────────────────────────────────────────────
df = pd.read_csv("../Dataset/Augmented_Cleaned_Data.csv")
print(f"✅ Loaded {len(df)} rows from Augmented_Cleaned_Data.csv\n")

decoded = pd.DataFrame()

# ── 1. Ordinal → original category strings ────────────────────────────────────

place_map = {0: "🏙️ Big metro city", 1: "🏢 Medium-sized city", 2: "🏘️ Small town", 3: "🌾 Rural area"}
importance_map = {0: "Not important", 1: "Slightly important", 2: "Very important"}

decoded["Place_Grew_Up"]     = df["Place_Grew_Up"].round().astype(int).map(place_map)
decoded["Price_Importance"]  = df["Price_Importance"].round().astype(int).map(importance_map)
decoded["Brand_Importance"]  = df["Brand_Importance"].round().astype(int).map(importance_map)
decoded["Peer_Importance"]   = df["Peer_Importance"].round().astype(int).map(importance_map)
decoded["Utility_Importance"]= df["Utility_Importance"].round().astype(int).map(importance_map)

# ── 2. OHE Track → single Track_Expenditures string ──────────────────────────

track_cols = ["Track_Bank_Balance", "Track_None", "Track_Payment_Apps", "Track_Apps_Spreadsheet"]
track_labels = {
    "Track_Bank_Balance":       "I check my bank balance occasionally.",
    "Track_None":               "I do not keep the track",
    "Track_Payment_Apps":       "I review my history within payment apps (e.g., UPI, Paytm).",
    "Track_Apps_Spreadsheet":   "I use a dedicated expense-tracking app or spreadsheet.",
}

def decode_track(row):
    for col in track_cols:
        if row[col] == 1:
            return track_labels[col]
    return "I check my bank balance occasionally."  # fallback

decoded["Track_Expenditures"] = df[track_cols].apply(decode_track, axis=1)

# ── 3. OHE Graph → single Expenditure_Graph string ───────────────────────────

graph_cols = ["Graph_Irregular_Random", "Graph_Spike_Then_Low", "Graph_High_Weekends", "Graph_Uniform_Daily"]
graph_labels = {
    "Graph_Irregular_Random":  "Irregular and Random Spending",
    "Graph_Spike_Then_Low":    "Spend a lot once and then low spending for rest",
    "Graph_High_Weekends":     "Steady Weekdays with High Weekends",
    "Graph_Uniform_Daily":     "Uniform Daily Expenses",
}

def decode_graph(row):
    for col in graph_cols:
        if row[col] == 1:
            return graph_labels[col]
    return "Uniform Daily Expenses"  # fallback

decoded["Expenditure_Graph"] = df[graph_cols].apply(decode_graph, axis=1)

# ── 4. JUE binary → Justify_Unexpected_Expense string ────────────────────────

jue_map = {
    "Emergency(JBS)":   "Emergencies (e.g., phone/laptop repair).",
    "Discounts(JBS)":   "A 50% discount on a brand I highly value.",
    "Party(JBS)":       "Social celebrations or parties.",
    "Workshop(JBS)":    "Skill development (workshops, certifications, technical kits).",
    "Trip(JBS)":        "A planned trip with friends.",
}

def decode_jue(row):
    selected = [label for col, label in jue_map.items() if row[col] == 1]
    return ", ".join(selected) if selected else "None"

decoded["Justify_Unexpected_Expense"] = df[list(jue_map.keys())].apply(decode_jue, axis=1)

# ── 5. MinMaxScaler inverse → back to 1–5 integers ───────────────────────────
# MMS was fit on range [1,5] so inverse is: round(val * 4 + 1)

def inverse_mms(val):
    return int(np.round(np.clip(val * 4 + 1, 1, 5)))

decoded["Unplanned_Purchases"] = df["Unplanned_Purchases"].apply(inverse_mms)
decoded["Peer_Influence"]      = df["Peer_Influence"].apply(inverse_mms)
decoded["Finance_Confidence"]  = df["Finance_Confidence"].apply(inverse_mms)

# ── 6. Monthly Spend & Group (already correct) ────────────────────────────────
decoded["Monthly_Spend"] = df["Monthly_Spend"].round().astype(int)
decoded["Group"]         = df["Group"].astype(str)

# ── 7. Budget binary → "Yes"/"No" strings (K-Prototypes needs categorical) ───
budget_col_map = {
    "Budget_FoodDining":    "Budget_FoodDining",
    "Budget_Travel":        "Budget_Travel",
    "Budget_Fashion":       "Budget_Fashion",
    "Budget_Subscriptions": "Budget_Subscriptions",
    "Budget_Entertainment": "Budget_Entertainment",
}

for raw_col, out_col in budget_col_map.items():
    decoded[out_col] = df[raw_col].apply(lambda x: "Yes" if int(round(x)) == 1 else "No")

# ── Quality check ─────────────────────────────────────────────────────────────
print("=== Decoded Quality Check ===")
print("\nPlace_Grew_Up distribution:")
print(decoded["Place_Grew_Up"].value_counts().to_dict())

print("\nTrack_Expenditures distribution:")
print(decoded["Track_Expenditures"].value_counts().to_dict())

print("\nExpenditure_Graph distribution:")
print(decoded["Expenditure_Graph"].value_counts().to_dict())

print("\nUnplanned_Purchases value counts:")
print(decoded["Unplanned_Purchases"].value_counts().sort_index().to_dict())

print("\nMonthly_Spend distribution:")
print(decoded["Monthly_Spend"].value_counts().sort_index().to_dict())

# ── Save ──────────────────────────────────────────────────────────────────────
decoded.to_csv("../Dataset/Augmented_Categorical_Data.csv", index=False)
print(f"\n🎉 Saved Augmented_KProto_Data.csv with {len(decoded)} rows!")
print("👉 Use this file as input for your K-Prototypes notebook.")


✅ Loaded 507 rows from Augmented_Cleaned_Data.csv

=== Decoded Quality Check ===

Place_Grew_Up distribution:
{'🏢 Medium-sized city': 280, '🏘️ Small town': 105, '🏙️ Big metro city': 102, '🌾 Rural area': 20}

Track_Expenditures distribution:
{'I review my history within payment apps (e.g., UPI, Paytm).': 244, 'I do not keep the track': 128, 'I check my bank balance occasionally.': 80, 'I use a dedicated expense-tracking app or spreadsheet.': 55}

Expenditure_Graph distribution:
{'Irregular and Random Spending': 266, 'Steady Weekdays with High Weekends': 106, 'Spend a lot once and then low spending for rest': 73, 'Uniform Daily Expenses': 62}

Unplanned_Purchases value counts:
{1: 68, 2: 120, 3: 177, 4: 77, 5: 65}

Monthly_Spend distribution:
{1: 38, 2: 66, 3: 71, 4: 85, 5: 53, 6: 28, 7: 12, 8: 19, 9: 50, 10: 85}

🎉 Saved Augmented_KProto_Data.csv with 507 rows!
👉 Use this file as input for your K-Prototypes notebook.
